In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
import pandas as pd
import joblib
from scipy.spatial import cKDTree

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Random Forest 

In [70]:
# 1. Load model
soil_properties = ["pH", "CE", "CEC", "OM", "A%", "L%", "S%"]
cv_method ='sp2_k5'
soil_prop = "S%"  # Change this to the desired soil property
model = joblib.load(f"../Models/RandomForest/sp2_k5/best_RandomForest_sp2_k5_all_depth_{soil_prop}_Without.joblib")
inputs = np.load("inputs_points_flat.npy")

In [71]:
from tqdm import tqdm

def batch_generator(X, batch_size=1024):
    N = X.shape[0]
    for i in range(0, N, batch_size):
        yield X[i:i+batch_size]

# Example: memory-efficient prediction
preds_list = []
batch_size = 1024  # adjust smaller if needed

for batch in tqdm(batch_generator(inputs, batch_size=batch_size), total=int(np.ceil(inputs.shape[0]/batch_size))):
    preds = model.predict(batch)
    preds_list.append(preds)

# Concatenate all batches into final predictions
predictions = np.concatenate(preds_list, axis=0)
print(predictions.shape)

  0%|          | 0/2758 [00:00<?, ?it/s]c:\Users\ASUS\anaconda3\envs\dsm_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\ASUS\anaconda3\envs\dsm_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\ASUS\anaconda3\envs\dsm_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\ASUS\anaconda3\envs\dsm_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
  0%|          | 4/2758 [00:00<01:14, 37.02it/s]c:\Users\ASUS\anaconda3\envs\dsm_env\lib\site-packages\sklearn\u

(2823300,)


In [72]:
# make it as dataframe 

soil_properties = [soil_prop]

df_predictions = pd.DataFrame(
    data=predictions,
    columns=soil_properties
)

df_predictions.describe()

,S%
count,2.823300e+06
mean,7.725568e+01
std,1.519654e+01
min,3.190694e+00
25%,7.874299e+01
50%,8.299154e+01
75%,8.558750e+01
max,9.703619e+01


In [73]:
neg_percent = (df_predictions[soil_properties] < 0).mean() * 100

print(neg_percent)

S%    0.0
dtype: float64


In [74]:
df_xy = pd.read_csv("points.csv")
df_joined = pd.concat([df_xy[['X', 'Y']], df_predictions], axis=1)
df_joined

,X,Y,S%
0,3.343779,18.967783,79.829008
1,3.352772,18.967783,79.091339
2,3.361765,18.967783,77.614293
3,3.370759,18.967783,78.402195
4,3.379752,18.967783,79.585977
...,...,...,...
2823295,8.937562,37.538783,45.776011
2823296,8.946555,37.538783,43.883461
2823297,8.946555,37.547776,45.655976
2823298,8.946555,37.556769,38.285893


In [75]:
df_joined.to_csv(f'predictions/RandomForest/rf_predictions_{soil_prop}.csv')

# LGBM

In [3]:
# 1. Load model
cv_method ='sp2_k5'
soil_prop = "CEC"  # Change this to the desired soil property
model = joblib.load(f"../Models/all/LGBM/sp2_k5/best_LGBM_sp2_k5_all_depth_{soil_prop}_Without.joblib")
inputs = np.load("inputs_points_flat.npy")

In [4]:
from tqdm import tqdm

def batch_generator(X, batch_size=1024):
    N = X.shape[0]
    for i in range(0, N, batch_size):
        yield X[i:i+batch_size]

# Example: memory-efficient prediction
preds_list = []
batch_size = 1024  # adjust smaller if needed

for batch in tqdm(batch_generator(inputs, batch_size=batch_size), total=int(np.ceil(inputs.shape[0]/batch_size))):
    preds = model.predict(batch)
    preds_list.append(preds)

# Concatenate all batches into final predictions
predictions = np.concatenate(preds_list, axis=0)
print(predictions.shape)

  0%|          | 0/2758 [00:00<?, ?it/s]

c:\Users\ASUS\anaconda3\envs\dsm_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\ASUS\anaconda3\envs\dsm_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\ASUS\anaconda3\envs\dsm_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\ASUS\anaconda3\envs\dsm_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\ASUS\anaconda3\envs\dsm_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature nam

(2823300,)


In [6]:
# make it as dataframe 

soil_properties = [soil_prop]

df_predictions = pd.DataFrame(
    data=predictions,
    columns=soil_properties
)

df_predictions.describe()

,CEC
count,2.823300e+06
mean,3.337421e+00
std,1.357870e+00
min,-3.137115e+00
25%,2.642195e+00
50%,3.391920e+00
75%,4.081179e+00
max,2.271132e+01


In [7]:
neg_percent = (df_predictions[soil_properties] < 0).mean() * 100

print(neg_percent)

CEC    0.890129
dtype: float64


In [8]:
df_xy = pd.read_csv("points.csv")
df_joined = pd.concat([df_xy[['X', 'Y']], df_predictions], axis=1)
df_joined

,X,Y,CEC
0,3.343779,18.967783,4.363877
1,3.352772,18.967783,3.409175
2,3.361765,18.967783,3.756508
3,3.370759,18.967783,3.574541
4,3.379752,18.967783,3.850449
...,...,...,...
2823295,8.937562,37.538783,-0.182068
2823296,8.946555,37.538783,-0.101252
2823297,8.946555,37.547776,-0.445417
2823298,8.946555,37.556769,3.353516


In [10]:
df_joined.to_csv(f'predictions/LGBM/lgbm_predictions_{soil_prop}.csv')

# MLP

In [37]:
#  improve predictions
#  
import pandas as pd

# Counters
stats = {
    "negative_found": 0,
    "over_100_found": 0,
    "total_zero": 0,
    "normalized": 0
}

def fix_texture(row):
    A_raw, L_raw, S_raw = row["A%"], row["L%"], row["S%"]

    #### 1. Detect negatives ####
    if A_raw < 0 or L_raw < 0 or S_raw < 0:
        stats["negative_found"] += 1

    #### 2. Detect values > 100 ####
    if A_raw > 100 or L_raw > 100 or S_raw > 100:
        stats["over_100_found"] += 1

    # Clip negatives to 0
    A = max(A_raw, 0)
    L = max(L_raw, 0)
    S = max(S_raw, 0)

    # Clip >100
    A = min(A, 100)
    L = min(L, 100)
    S = min(S, 100)

    total = A + L + S

    #### 3. Detect total = 0 ####
    if total == 0:
        stats["total_zero"] += 1
        return pd.Series([33.33, 33.33, 33.33])

    #### 4. Normal renormalization ####
    stats["normalized"] += 1
    A = (A / total) * 100
    L = (L / total) * 100
    S = (S / total) * 100

    return pd.Series([A, L, S])


df_joined[["A%_impr", "L%_impr", "S%_impr"]] = df_joined.apply(fix_texture, axis=1)

print("\n===== SCENARIO SUMMARY =====")
print(f"Rows with NEGATIVE values:        {stats['negative_found']}")
print(f"Rows with values > 100:          {stats['over_100_found']}")
print(f"Rows where total was ZERO:       {stats['total_zero']}")
print(f"Rows renormalized normally:      {stats['normalized']}")
print("================================\n")



===== SCENARIO SUMMARY =====
Rows with NEGATIVE values:        17835
Rows with values > 100:          38533
Rows where total was ZERO:       0
Rows renormalized normally:      2823300



In [38]:
df_joined

,X,Y,pH,CE,CEC,OM,A%,L%,S%,A%_impr,L%_impr,S%_impr
0,3.343779,18.967783,7.706077,1.251239,2.236153,-0.238614,8.162644,6.231542,103.490829,7.135541,5.447429,87.417030
1,3.352772,18.967783,7.730124,1.301586,4.165951,0.030967,9.024846,8.223270,95.215569,8.024676,7.311934,84.663390
2,3.361765,18.967783,7.723734,0.896246,4.374993,-0.022386,8.507742,8.109316,95.301956,7.601695,7.245700,85.152605
3,3.370759,18.967783,7.728781,1.041527,4.501424,-0.016235,8.464398,8.600489,94.182823,7.608605,7.730935,84.660460
4,3.379752,18.967783,7.747868,1.509896,4.510503,0.076005,8.752970,9.624228,92.038071,7.927318,8.716392,83.356290
...,...,...,...,...,...,...,...,...,...,...,...,...
2823295,8.937562,37.538783,7.956028,14.996037,0.720112,2.205137,14.542761,37.732368,47.799355,14.531937,37.704285,47.763778
2823296,8.946555,37.538783,7.458802,22.589920,1.027723,2.020257,28.198763,37.060307,34.647758,28.225061,37.094869,34.680071
2823297,8.946555,37.547776,7.435527,15.958015,1.519760,2.031544,19.173237,33.510159,46.296646,19.370811,33.855470,46.773719
2823298,8.946555,37.556769,7.670339,14.537575,1.914330,2.003577,21.187021,35.281338,42.530418,21.401296,35.638155,42.960549


In [41]:
# for each of props having negative values 

soil_props_with_neg = ['CE',	'CEC',	'OM']
points = df_joined[['X', 'Y']].values

for prop in soil_props_with_neg:
    values = df_joined[prop].values
    valid_idx = np.where(values > 0)[0]
    high_idx = np.where(values < 0)[0]

    tree = cKDTree(points[valid_idx])
    distances, nearest_idx = tree.query(points[high_idx], k=1)
    values[high_idx] = values[valid_idx[nearest_idx]]

    # Add the improv

    df_joined[f'{prop}_impr'] = values


In [42]:
df_joined

,X,Y,pH,CE,CEC,OM,A%,L%,S%,A%_impr,L%_impr,S%_impr,CE_impr,CEC_impr,OM_impr
0,3.343779,18.967783,7.706077,1.251239,2.236153,0.030967,8.162644,6.231542,103.490829,7.135541,5.447429,87.417030,1.251239,2.236153,0.030967
1,3.352772,18.967783,7.730124,1.301586,4.165951,0.030967,9.024846,8.223270,95.215569,8.024676,7.311934,84.663390,1.301586,4.165951,0.030967
2,3.361765,18.967783,7.723734,0.896246,4.374993,0.030967,8.507742,8.109316,95.301956,7.601695,7.245700,85.152605,0.896246,4.374993,0.030967
3,3.370759,18.967783,7.728781,1.041527,4.501424,0.274955,8.464398,8.600489,94.182823,7.608605,7.730935,84.660460,1.041527,4.501424,0.274955
4,3.379752,18.967783,7.747868,1.509896,4.510503,0.076005,8.752970,9.624228,92.038071,7.927318,8.716392,83.356290,1.509896,4.510503,0.076005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2823295,8.937562,37.538783,7.956028,14.996037,0.720112,2.205137,14.542761,37.732368,47.799355,14.531937,37.704285,47.763778,14.996037,0.720112,2.205137
2823296,8.946555,37.538783,7.458802,22.589920,1.027723,2.020257,28.198763,37.060307,34.647758,28.225061,37.094869,34.680071,22.589920,1.027723,2.020257
2823297,8.946555,37.547776,7.435527,15.958015,1.519760,2.031544,19.173237,33.510159,46.296646,19.370811,33.855470,46.773719,15.958015,1.519760,2.031544
2823298,8.946555,37.556769,7.670339,14.537575,1.914330,2.003577,21.187021,35.281338,42.530418,21.401296,35.638155,42.960549,14.537575,1.914330,2.003577


In [43]:
df_joined.to_csv('mlp_predictions_improved.csv')

In [30]:
print("Min prediction:", predictions.min())
print("Max prediction:", predictions.max())

Min prediction: -14.547633
Max prediction: 192.62685


In [ ]:
predictions = predictions.flatten()

In [ ]:
# df has columns: X, Y, pH_value
points = df[['X', 'Y']].values
values = df['S%_value'].values

# Find indices where pH < 4 and pH >= 4
valid_idx = np.where(values <= 100)[0]
high_idx = np.where(values > 100)[0]

# Build KDTree for points with pH >= 4
tree = cKDTree(points[valid_idx])

# For each low pH point, find nearest high pH point
distances, nearest_idx = tree.query(points[high_idx], k=1)

# Replace low values with the corresponding nearest high pH value
values[high_idx] = values[valid_idx[nearest_idx]]
df = df.drop(columns=["S%_value"])
# Update DataFrame
df['S%_values'] = values

In [ ]:
# Quick check of pH range
min_S = df['S%_values'].min()
max_S = df['S%_values'].max()

print(f"Minimum S value: {min_S}")
print(f"Maximum S value: {max_S}")


Minimum S value: 0.0002269744873046875
Maximum S value: 99.99971008300781


In [ ]:
df.to_csv('./predictions/S%.csv', index=False)